In [ ]:
import torch
import torch.nn as nn
import numpy as np
import torchvision
from torchvision.transforms import v2 as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torch.functional import F
import torch.optim as optim
from rich.progress import track

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

In [ ]:
model.fc = nn.Linear(model.fc.in_features,10)

In [ ]:
for param in model.parameters():
  param.requires_grad = False

for param in model.fc.parameters():
  param.requires_grad = True

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
train_data = ImageFolder(root='/content/drive/MyDrive/train', transform=transform)
test_data = ImageFolder(root='/content/drive/MyDrive/test', transform=transform)

In [ ]:
train_loader = DataLoader(train_data,batch_size=10,shuffle=True)
test_loader = DataLoader(test_data,batch_size=10,shuffle=True)

In [ ]:
model.conv1 = nn.Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in track(range(100),description="Training Images: "):
  for image,label in train_loader:
    outputs = model(image)
    # Removed: _, preds = torch.max(outputs,1)

    loss = loss_fn(outputs,label)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()
  print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

In [ ]:
correct_predictions = 0
total_predictions = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = 100 * correct_predictions / total_predictions
print(f'Accuracy of the model on the {total_predictions} test images: {accuracy:.2f}%')